# VLA Foundry Data Visualization

This notebook demonstrates how to visualize robotics datasets using VLA Foundry's visualization tools built on [Rerun](https://rerun.io/).

## Prerequisites

Before running this notebook, ensure you have:
1. Installed the VLA Foundry kernel: `bash tutorials/install_kernel.sh`
2. Selected the "Python (vla_foundry)" kernel in Jupyter

## Overview

The visualization pipeline:
1. Downloads a sample dataset (PickAndPlaceBox)
2. Runs the visualization script to generate Rerun visualization
3. Displays the interactive 3D viewer inline in the notebook

## Configuration

Set the task name and visualization parameters:

In [ ]:
# Dataset configuration
TASK_NAME = "PickAndPlaceBox"
DATA_PATH = f"data/{TASK_NAME}"

# Visualization parameters
NUM_EPISODES = 3  # Number of episodes to visualize
SUBSAMPLE_FACTOR = 10  # Visualize every Nth sample (higher = faster, less data)

## Setup

Initialize the Rerun visualizer backend:

In [ ]:
import os
import subprocess

# Change to the repo root so all downstream paths are relative.
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

# Select the Rerun visualizer backend.
os.environ["VISUALIZER"] = "rerun"

## Download Sample Dataset

Download the PickAndPlaceBox dataset for visualization:

> ⚠️ **Heavy download ahead.** The next cell downloads PickAndPlaceBox (~1.3 GB) from S3. The step is idempotent (safe to re-run; skips files already present). Under `jupyter nbconvert`, pass `--ExecutePreprocessor.timeout=900` or pre-download from a terminal.


In [ ]:
import subprocess

subprocess.run(
    [
        "python",
        "vla_foundry/data/scripts/download_dataset.py",
        "--task",
        TASK_NAME,
        "--local_path",
        f"tutorials/{DATA_PATH}",
    ],
    check=True,
)

## Visualize Dataset

The visualization invokes `vla_foundry.data.scripts.vis.lbm_vis.main()` directly to generate a Rerun visualization that displays inline in the notebook.

In [ ]:
import json
import sys
from pathlib import Path

import rerun as rr
import yaml

# Resolve dataset path (the download cell wrote into tutorials/data/).
# Metadata (manifest/stats/preprocessing_config) live at the dataset root;
# only the tar shards are under shards/.
dataset_path = Path("tutorials") / DATA_PATH
manifest_path = dataset_path / "manifest.jsonl"
stats_path = dataset_path / "stats.json"
preprocessing_config_path = dataset_path / "preprocessing_config.yaml"

# Sum num_sequences across the first NUM_EPISODES manifest entries
with open(manifest_path) as f:
    manifest_entries = [json.loads(line) for line in f][:NUM_EPISODES]
num_samples = sum(entry["num_sequences"] for entry in manifest_entries)

# Read camera/image config from the dataset's preprocessing_config.yaml
with open(preprocessing_config_path) as f:
    preprocessing_cfg = yaml.safe_load(f)

camera_names = preprocessing_cfg["camera_names"]
image_indices = preprocessing_cfg["image_indices"]
# Generate image names with timestep suffixes (e.g., "scene_right_0_t-1", "scene_right_0_t0")
image_names = [f"{c}_t{idx}" for c in camera_names for idx in image_indices]

# Build draccus args mirroring the previous shell-wrapper behavior
sys.argv = [
    "lbm_vis.py",
    "--ordered",
    f"--subsample={SUBSAMPLE_FACTOR}",
    "--config_path=examples/visualization/visualization_params.yaml",
    f"--data.dataset_manifest=[{manifest_path}]",
    f"--data.dataset_statistics=[{stats_path}]",
    f"--data.camera_names=[{','.join(camera_names)}]",
    f"--data.image_indices=[{','.join(str(i) for i in image_indices)}]",
    f"--data.image_names=[{','.join(image_names)}]",
    "--data.shuffle=true",
    f"--total_train_samples={num_samples}",
    "--num_checkpoints=1",
    "--data.num_workers=0",
    "--data.normalization.enabled=false",
    "--data.normalization.field_configs={}",
]

from vla_foundry.data.scripts.vis.lbm_vis import main  # noqa: E402

main()

rr.notebook_show()